# Lab Task 2: Car Safety/Evaluation Prediction (Decision Tree Classifier)
Runs directly in Google Colab: *Runtime > Run all*. Data is downloaded automatically from the UCI Machine Learning Repository (Car Evaluation Data Set).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ---- Load data ----
cols = ["buying", "maint", "doors", "persons", "lug_boot", "safety", "class"]
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data"
try:
    df = pd.read_csv(url, header=None, names=cols)
except Exception:
    from google.colab import files
    up = files.upload()   # upload car.data downloaded from UCI
    df = pd.read_csv(list(up.keys())[0], header=None, names=cols)

print(df.shape)
df.head()

In [ ]:
# ---- Explore ----
print(df.isnull().sum())
print(df["class"].value_counts())
for c in cols[:-1]:
    print(c, df[c].unique())

In [ ]:
# ---- Encode categorical features (ordered categories) ----
order = [
    ["low", "med", "high", "vhigh"],      # buying
    ["low", "med", "high", "vhigh"],      # maint
    ["2", "3", "4", "5more"],             # doors
    ["2", "4", "more"],                   # persons
    ["small", "med", "big"],              # lug_boot
    ["low", "med", "high"],               # safety
]
X = pd.DataFrame(OrdinalEncoder(categories=order).fit_transform(df[cols[:-1]]), columns=cols[:-1])
y = df["class"]   # unacc, acc, good, vgood

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# ---- Train Decision Tree ----
clf = DecisionTreeClassifier(criterion="entropy", max_depth=6, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print("Training accuracy:", clf.score(X_train, y_train))
print("Test accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\n", classification_report(y_test, y_pred))

In [ ]:
# ---- Visualize ----
plt.figure(figsize=(22, 10))
plot_tree(clf, feature_names=X.columns, class_names=clf.classes_, filled=True, rounded=True, fontsize=8)
plt.show()

pd.Series(clf.feature_importances_, index=X.columns).sort_values().plot(kind="barh", title="Feature importance")
plt.show()

In [ ]:
# ---- Predict a new car ----
# buying, maint, doors, persons, lug_boot, safety
sample = pd.DataFrame([["low", "low", "4", "4", "big", "high"]], columns=cols[:-1])
sample_enc = pd.DataFrame(OrdinalEncoder(categories=order).fit(df[cols[:-1]]).transform(sample), columns=cols[:-1])
print("Predicted class:", clf.predict(sample_enc)[0])